In [ ]:
# Fix: Kaggle's default NumPy 1.26.4 is CORRECT - prevent upgrades to 2.x
import sys
print("Step 1: Record Kaggle's default NumPy version...")
import numpy as np
default_numpy = np.__version__
print(f"   Default NumPy: {default_numpy}")

print("\nStep 2: Install ultralytics...")
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print("\nStep 3: LOCK NumPy back to Kaggle's default...")
subprocess.run(["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"], check=True)

print("\nInstallation complete!")
import numpy as np
from ultralytics import YOLO
print(f"   NumPy: {np.__version__}")
print("   OK Ultralytics imported successfully")

In [ ]:
import json
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

print("OK All modules imported successfully")

In [ ]:
# Configure paths for Kaggle
if os.path.exists("/kaggle/input"):
    IN_KAGGLE = True
    BASE_INPUT_PATH = Path("/kaggle/input")
    OUTPUT_PATH = Path("/kaggle/working")
    DATASET_PATH = BASE_INPUT_PATH / "baseline-outputs/data_wound_seg"
else:
    IN_KAGGLE = False
    BASE_INPUT_PATH = PROJECT_ROOT / "data"
    OUTPUT_PATH = PROJECT_ROOT / "outputs"
    DATASET_PATH = BASE_INPUT_PATH / "baseline-outputs/data_wound_seg"

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Output path: {OUTPUT_PATH}")

# Verify dataset exists
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

print("OK Environment configured")

## Test-Time Augmentation (TTA) Smoke Test

**Intervention**: Ensemble predictions from multiple augmentations (h-flip, v-flip, rotate 90°)

This smoke test validates:
1. Augmentation transforms (forward and inverse)
2. Mask ensemble logic
3. TTA improves or maintains Dice score vs baseline

In [ ]:
import torch

# Define TTA augmentations (4 core transforms for smoke test)
AUGMENTATIONS = {
    "original": (
        lambda x: x,
        lambda x: x
    ),
    "horizontal_flip": (
        lambda x: torch.flip(x, dims=[-1]),
        lambda x: torch.flip(x, dims=[-1])
    ),
    "vertical_flip": (
        lambda x: torch.flip(x, dims=[-2]),
        lambda x: torch.flip(x, dims=[-2])
    ),
    "rotate_90": (
        lambda x: torch.rot90(x, k=1, dims=[-2, -1]),
        lambda x: torch.rot90(x, k=-1, dims=[-2, -1])
    ),
}

print(f"OK Defined {len(AUGMENTATIONS)} augmentations for TTA")

In [ ]:
# Test forward-inverse identity (augmentations must be perfect inverses)
test_tensor = torch.rand(3, 512, 512)

for aug_name in AUGMENTATIONS:
    forward_fn, inverse_fn = AUGMENTATIONS[aug_name]
    transformed = forward_fn(test_tensor)
    reconstructed = inverse_fn(transformed)
    error = torch.abs(reconstructed - test_tensor).max().item()
    
    if error > 1e-6:
        raise AssertionError(f"{aug_name}: forward-inverse error = {error:.2e}")
    
    print(f"OK {aug_name}: forward-inverse error = {error:.2e}")

print("\nOK All augmentation transforms are valid inverses")

In [ ]:
# Create YOLO dataset configuration
config_path = OUTPUT_PATH / "wound_dataset.yaml"
config_content = f"""
path: {DATASET_PATH.parent}
train: data_wound_seg/train
val: data_wound_seg/test

names:
  0: wound
"""

with open(config_path, 'w') as f:
    f.write(config_content.strip())

print(f"OK YOLO config created: {config_path}")

In [ ]:
# Train baseline model on tiny dataset (1 epoch for speed)
print("Training baseline model...")

model = YOLO("yolov8n-seg.pt")
results = model.train(
    data=str(config_path),
    epochs=1,  # Smoke test - just verify training works
    imgsz=512,
    batch=8,
    workers=2,
    seed=42,
    save=True,
    project=str(OUTPUT_PATH / "baseline_training"),
    name="smoke_test",
    exist_ok=True,
    verbose=False,
)

print("OK Training complete")

# Load trained model
MODEL_PATH = OUTPUT_PATH / "baseline_training/smoke_test/weights/best.pt"
model = YOLO(str(MODEL_PATH))
print(f"OK Loaded trained model from {MODEL_PATH}")


In [ ]:
def extract_mask_inline(result, target_shape):
    """Extract mask from YOLO result inline (no external imports)."""
    if result.masks is not None and len(result.masks) > 0:
        mask_data = result.masks.data[0].cpu().numpy()
        if mask_data.shape != target_shape:
            mask_data = cv2.resize(mask_data, (target_shape[1], target_shape[0]))
        return (mask_data > 0.5).astype(np.uint8)
    return np.zeros(target_shape, dtype=np.uint8)

def predict_with_tta(model, image_path, augmentation_set=None, aggregation="mean"):
    """Predict with Test-Time Augmentation."""
    if augmentation_set is None:
        augmentation_set = AUGMENTATIONS

    # Load image
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    predictions = []

    for aug_name in augmentation_set:
        forward_fn, inverse_fn = augmentation_set[aug_name]

        # Convert to tensor for augmentation
        img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float()

        # Apply augmentation
        aug_img_tensor = forward_fn(img_tensor)

        # Convert back to numpy for YOLO
        aug_img = aug_img_tensor.permute(1, 2, 0).numpy().astype(np.uint8)

        # Predict
        result = model.predict(aug_img, verbose=False)[0]
        pred_mask = extract_mask_inline(result, img_rgb.shape[:2])

        # Convert mask to tensor for inverse transform
        pred_tensor = torch.from_numpy(pred_mask).float()
        pred_reversed = inverse_fn(pred_tensor)
        predictions.append(pred_reversed.numpy())

    # Aggregate predictions
    if aggregation == "mean":
        ensemble_mask = (np.mean(predictions, axis=0) > 0.5).astype(np.uint8)
    elif aggregation == "majority_vote":
        ensemble_mask = (np.sum(predictions, axis=0) > (len(predictions) / 2)).astype(np.uint8)
    else:
        raise ValueError(f"Unknown aggregation: {aggregation}")

    return ensemble_mask

print("OK TTA prediction function defined")

In [ ]:
# Test on 3 images (smoke test)
import time

def calculate_dice(pred_mask, gt_mask):
    """Calculate Dice coefficient inline."""
    pred_mask = (pred_mask > 0).astype(np.uint8)
    gt_mask = (gt_mask > 0).astype(np.uint8)
    intersection = np.sum(pred_mask * gt_mask)
    union = np.sum(pred_mask) + np.sum(gt_mask)
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return 2.0 * intersection / union

TEST_IMAGES_DIR = DATASET_PATH / "test_images"
TEST_MASKS_DIR = DATASET_PATH / "test_masks"
test_images = sorted(TEST_IMAGES_DIR.glob("*.png"))[:3]

baseline_results = []
tta_results = []
baseline_times = []
tta_times = []

for img_path in test_images:
    image_id = img_path.stem
    mask_path = TEST_MASKS_DIR / f"{image_id}.png"

    if not mask_path.exists():
        print(f"WARNING  Skipping {image_id} - no ground truth")
        continue

    # Load ground truth
    gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    # Baseline prediction
    start = time.time()
    img_rgb = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)
    baseline_result = model.predict(img_rgb, verbose=False)[0]
    baseline_mask = extract_mask_inline(baseline_result, img_rgb.shape[:2])
    baseline_times.append(time.time() - start)

    # TTA prediction
    start = time.time()
    tta_mask = predict_with_tta(model, img_path, aggregation="mean")
    tta_times.append(time.time() - start)

    # Calculate metrics
    baseline_dice = calculate_dice(baseline_mask, gt_mask)
    tta_dice = calculate_dice(tta_mask, gt_mask)

    baseline_results.append({'image_id': image_id, 'dice': baseline_dice})
    tta_results.append({'image_id': image_id, 'dice': tta_dice})

    print(f"\n{image_id}:")
    print(f"  Baseline: Dice={baseline_dice:.4f} ({baseline_times[-1]:.3f}s)")
    print(f"  TTA:      Dice={tta_dice:.4f} ({tta_times[-1]:.3f}s)")

print(f"\nOK Evaluated {len(baseline_results)} images")

In [ ]:
# Compare aggregated metrics
baseline_df = pd.DataFrame(baseline_results)
tta_df = pd.DataFrame(tta_results)

baseline_dice_mean = baseline_df['dice'].mean()
tta_dice_mean = tta_df['dice'].mean()
dice_improvement = tta_dice_mean - baseline_dice_mean

print("\n=== Baseline vs TTA (3 images) ===")
print(f"Baseline Dice: {baseline_dice_mean:.4f} ± {baseline_df['dice'].std():.4f}")
print(f"TTA Dice:      {tta_dice_mean:.4f} ± {tta_df['dice'].std():.4f}")
print(f"Improvement:   {dice_improvement:+.4f} ({dice_improvement/baseline_dice_mean*100:+.2f}%)")
print(f"\nBaseline time: {np.mean(baseline_times):.3f}s per image")
print(f"TTA time:      {np.mean(tta_times):.3f}s per image")

In [ ]:
# Validation checks
smoke_test_report = {
    "ablation": "6",
    "intervention": "Test-Time Augmentation (TTA)",
    "augmentations": list(AUGMENTATIONS.keys()),
    "validation_checks": {
        "augmentation_transforms": {"passed": True},
        "forward_inverse_identity": {"passed": True},
        "tta_dice_maintained": {
            "passed": bool(tta_dice_mean >= baseline_dice_mean - 0.01),  # Convert NumPy bool to Python bool
            "baseline_dice": float(baseline_dice_mean),
            "tta_dice": float(tta_dice_mean),
            "improvement": float(dice_improvement)
        },
    },
    "smoke_test_status": "PASSED" if tta_dice_mean >= baseline_dice_mean - 0.01 else "FAILED"
}

all_passed = all(
    check["passed"]
    for check in smoke_test_report["validation_checks"].values()
)

if not all_passed:
    raise AssertionError("Some validation checks failed!")

report_path = OUTPUT_PATH / "smoke_test_report.json"
with open(report_path, 'w') as f:
    json.dump(smoke_test_report, f, indent=2)

print(f"\nOK Smoke test report saved: {report_path}")
print("\nALL VALIDATIONS PASSED!")
print(f"TTA Dice improvement: {dice_improvement:+.4f}")

In [ ]:
print("\n" + "="*60)
print("SMOKE TEST REPORT")
print("="*60)
print(json.dumps(smoke_test_report, indent=2))